# v3 DPO Training: v2.2 from v2.1-lite + preference pairs

Trains DPO on top of v2.1-lite using the preference pairs from v3_pair_gen.
Exports v2.2 as GGUF q4_k_m for local evaluation.

**Problem targeted:** Problem 2 (behavioural misses on in-distribution data).
**Runs on:** Kaggle GPU (T4 or P100).
**Input:** `data/qwen3.5_latest/dpo_pairs.jsonl` (from v3_pair_gen output)
**Output:** `v2.2_dpo-qwen3b-q4_k_m.gguf`

Setup instructions before running:
1. Add v3_pair_gen output as input (contains dpo_pairs.jsonl).
2. Add v2-1-lite-multi-template-sft-unsloth output as input (contains merged_fp16_v2_1_lite).

## Cell 1: env check

In [ ]:
import sys
print('Python:', sys.version)
!nvidia-smi

## Cell 2: clone repo

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT  = UserSecretsClient().get_secret('GITHUB_PAT')
REPO = 'nizamphoenix/clinical_transcript_summariser'
DEST = '/kaggle/working/repo'

if os.path.exists(DEST):
    subprocess.run(['rm', '-rf', DEST], check=True)

url = f'https://{PAT}@github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, DEST], check=True)
print('cloned ->', DEST)
%cd $DEST

## Cell 3: install dependencies

In [ ]:
!pip install -q unsloth
!pip uninstall unsloth -y
!pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q -e .

import sys
sys.path.insert(0, '/kaggle/working/repo')
print('install done')

## Cell 4: load preference pairs

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

# dpo_pairs.jsonl is saved to the repo data dir by v3_pair_gen,
# but on Kaggle it may arrive via a separate input dataset.
# Try repo data dir first, then /kaggle/input.
PAIRS_CANDIDATES = [
    Path('data/qwen3.5_latest/dpo_pairs.jsonl'),
    Path('/kaggle/input/v3-pair-gen/dpo_pairs.jsonl'),  # adjust name if needed
]

pairs_path = next((p for p in PAIRS_CANDIDATES if p.exists()), None)
assert pairs_path is not None, (
    "dpo_pairs.jsonl not found. Add v3_pair_gen output as a Kaggle input dataset."
)
print('loading pairs from', pairs_path)

with open(pairs_path) as f:
    pairs = [json.loads(l) for l in f if l.strip()]

# DPOTrainer needs prompt, chosen, rejected columns
ds = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in pairs
])

print(f'pairs loaded: {len(ds)}')

# Summary
from collections import Counter
modes = Counter(p['rejected_mode'] for p in pairs)
tmpls = Counter(p['template'] for p in pairs)
print('by template:', dict(tmpls))
print('by rejected_mode:', dict(modes))

## Cell 5: load v2.1 base + fresh LoRA

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()

MODEL_PATH  = '/kaggle/input/v2-1-lite-multi-template-sft-unsloth/merged_fp16_v2_1_lite'
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
print('base loaded')

# Attach a fresh LoRA (separate from the SFT adapter — clean separation)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=0,
)
model.print_trainable_parameters()

## Cell 5b: sequence-length audit

Check what fraction of pairs would be truncated at `max_seq_length=1024`.
This matches the SFT baseline for fair comparison. Any pair exceeding 900 tokens
(90% of limit) is flagged — review the count before training.

In [ ]:
# Tokenise prompt+chosen and prompt+rejected to measure sequence lengths.
# DPOTrainer packs both into one forward pass; the longer of the two determines
# whether a pair is truncated.
# We check against max_seq_length=1024 (matching SFT baseline).
MAX_SEQ_LENGTH = 1024
WARN_THRESHOLD = 0.9  # flag pairs using >90% of budget

lengths = []
for p in pairs:
    combined_chosen   = p['prompt'] + p['chosen']
    combined_rejected = p['prompt'] + p['rejected']
    # Tokenise without adding special tokens to get raw length
    len_c = len(tokenizer.encode(combined_chosen,   add_special_tokens=False))
    len_r = len(tokenizer.encode(combined_rejected, add_special_tokens=False))
    lengths.append(max(len_c, len_r))

over_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
near_limit  = sum(1 for l in lengths if l > MAX_SEQ_LENGTH * WARN_THRESHOLD)
mean_len    = sum(lengths) / len(lengths)
max_len     = max(lengths)

print(f'pairs checked       : {len(lengths)}')
print(f'mean max-side length: {mean_len:.0f} tokens')
print(f'max  max-side length: {max_len} tokens')
print(f'pairs > {MAX_SEQ_LENGTH} (truncated): {over_limit}  ({100*over_limit/len(lengths):.1f}%)')
print(f'pairs > {int(MAX_SEQ_LENGTH*WARN_THRESHOLD)} (near limit): {near_limit}  ({100*near_limit/len(lengths):.1f}%)')

if over_limit > 0:
    print(f'\nWARNING: {over_limit} pairs will be silently truncated at {MAX_SEQ_LENGTH}.')
    print('These are consistent with the SFT training budget; no config change needed.')
    print('If >20% of pairs are truncated, consider raising max_seq_length and checking T4 OOM.')
else:
    print('\nAll pairs fit within max_seq_length=1024. No truncation risk.')


## Cell 6: DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig

# Conservative hyperparameters: low beta keeps KL anchor tight,
# preventing the model from trading misses for hallucinations.
config = DPOConfig(
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    seed=0,
    max_length=2048,
    max_prompt_length=1024,
    output_dir='sft_out_v2_2',
    logging_steps=10,
    save_strategy='no',
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    args=config,
    train_dataset=ds,
    tokenizer=tokenizer,
)

print('starting DPO training...')
trainer.train()
print('training complete')

## Cell 7: sanity check before export

Run the trained model on a few in-distribution transcripts and verify
`schema_valid=1` on all outputs. If any fail, investigate before exporting.

In [ ]:
from src.verifier import score_prediction
from src.prompts import build_inference_messages
from src.data_generation.templates import REGISTRY
import json

FastLanguageModel.for_inference(model)

DATA_ROOT = 'data/qwen3.5_latest'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Pick 2 samples per trained template
sanity_rows = []
for tpl, fname in [('soap', 'eval_in_dist.soap.jsonl'),
                   ('referral_a', 'eval_in_dist.referral_a.jsonl'),
                   ('mse', 'eval_in_dist.mse.jsonl')]:
    rows = load_jsonl(f'{DATA_ROOT}/{fname}')
    for r in rows[:2]:
        sanity_rows.append(dict(r, template=tpl))

print(f'sanity checking on {len(sanity_rows)} samples...')
print()
all_ok = True
for row in sanity_rows:
    template   = row['template']
    label_key  = REGISTRY[template]['label_key']
    gold       = row[label_key]
    transcript = row['transcript']

    messages = build_inference_messages(template, transcript)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=1024, temperature=0.0,
                         pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    sc = score_prediction(template, raw, gold, transcript)
    status = 'OK' if sc['schema_valid'] else 'FAIL'
    if not sc['schema_valid']:
        all_ok = False
    print(f'  [{status}] template={template}  schema_valid={sc["schema_valid"]}  '
          f'overlap={sc["content_overlap"]:.2f}  wrong_null={sc["wrong_null"]}/{sc["gold_filled"]}')

print()
if all_ok:
    print('All sanity checks passed. Safe to export.')
else:
    print('WARNING: schema_valid=0 on some outputs. Investigate before exporting.')

## Cell 8: merge LoRA and export GGUF q4_k_m

In [ ]:
# Merge LoRA into the base weights
model.save_pretrained_merged('merged_fp16_v2_2', tokenizer, save_method='merged_16bit')
print('merged weights saved -> merged_fp16_v2_2/')

# Export to GGUF q4_k_m
# Unsloth names the file: v2.2_dpo-unsloth.Q4_K_M.gguf
model.save_pretrained_gguf('v2.2_dpo', tokenizer, quantization_method='q4_k_m')
print('GGUF saved')
print()
print('Download the GGUF and rename to: v2.2_dpo-qwen3b-q4_k_m.gguf')
print('Place in: models/ in the local repo')
!ls -lh *.gguf 2>/dev/null || echo 'no gguf in cwd, check Kaggle output panel'